# EcoRewind 2.0 — Phase 1: data audit (Colab)

Reads the GeoTIFFs **directly from Google Drive**; nothing is downloaded to your laptop. Writes `reports/data_audit.md` to Drive under `MyDrive/EcoRewind2/reports/`.

Runtime: CPU is enough. Full resolution (`STRIDE=1`) takes roughly 15–30 min over ~95 files; `STRIDE=2` is ~4× less I/O-bound compute and fine for the audit.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Code comes from the zip you uploaded to Drive (MyDrive/EcoRewind2/ecorewind.zip).
ZIP = '/content/drive/MyDrive/EcoRewind2/ecorewind.zip'
!rm -rf /content/ecorewind && unzip -q "$ZIP" -d /content
!pip -q install rasterio==1.4.3
%cd /content/ecorewind

### Locate the three source folders
The v2 folder is titled `EcoRewind/Mississippi_v2` at the My Drive root; the `/` makes its mount path unusual, so we find it by listing.

In [ ]:
import os, glob
ROOT = '/content/drive/MyDrive'
v1 = f'{ROOT}/EcoRewind'
print('v1 Everglades files :', len(glob.glob(f'{v1}/Everglades/Everglades_*.tif')))
print('v1 Mississippi files:', len(glob.glob(f'{v1}/Mississippi/Mississippi_*.tif')))
cands = [os.path.join(ROOT, d) for d in os.listdir(ROOT) if 'Mississippi_v2' in d]
cands += [os.path.join(v1, d) for d in os.listdir(v1) if 'Mississippi_v2' in d]
for c in cands: print('v2 candidate:', repr(c), len(glob.glob(f'{c}/Mississippi_v2_*.tif')), 'tifs')
V2 = max(cands, key=lambda c: len(glob.glob(f'{c}/Mississippi_v2_*.tif')))
print('Using V2 =', V2)

### Smoke test (2 files per source)

In [ ]:
!python -m pytest -q tests
!python scripts/audit_data.py --source "mississippi_v2=$V2" --max-files 2 --stride 8 --out /content/smoke

### Full audit
Results cache per file in Drive, so if Colab disconnects just re-run this cell.

In [ ]:
OUT = '/content/drive/MyDrive/EcoRewind2/reports'
STRIDE = 1
!python scripts/audit_data.py --source "mississippi_v2=$V2" --stride $STRIDE --out "$OUT"

In [ ]:
from IPython.display import Markdown
Markdown(open(f'{OUT}/data_audit.md').read())